In [1]:
import wandb
import pandas as pd
import numpy as np

from utils import (
    WandbParser,
    create_main_table
)

api = wandb.Api()
wandb_parser = WandbParser(
    entity="bsarec",
    api=api,
    verbose=False,
    users=["hcbakker2"] # only load runs from this user
)
metrics = ["HR@5", "HR@10", "HR@20", "NDCG@5", "NDCG@10", "NDCG@20"]

# Reproduction

In [ ]:
def standard_post_processing(df: pd.DataFrame) -> pd.DataFrame:
    df = df.rename(columns={"model_type": "model_name", "data_name": "dataset"})
    df["dataset"] = df["dataset"].str.replace("_", " ")
    df["model_name"] = df["model_name"].str.replace("_", " ")
    df["metric"] = df["metric"].str.replace("test/", "")

    return df


models = ["SASRec", "BERT4Rec", "FMLPRec", "duoRec", "FeaRec", "BSARec"]
datasets = [
    'Beauty',
    'LastFM',
    'ML-1M',
    'Sports_and_Outdoors',
    'Toys_and_Games',
    'Yelp'
]

# retrieve reproduction runs
df = wandb_parser.register_and_parse(
    project="reproduction_experiments_new",
    constraints=dict(model_type=models, data_name=datasets),
    cfg=["model_type", "data_name", "seed"],
    summary=[f"test/{metric_name}" for metric_name in metrics],
    post_processing=standard_post_processing
)

main_table_latex, _, _ = create_main_table(
    df.copy(deep=True),
    models=models,
    metrics=metrics,
    target_model="BSARec",
    show_second_best=True
)

print(main_table_latex)

# Padding

In [ ]:
def standard_post_processing_padding(df: pd.DataFrame) -> pd.DataFrame:
    df = df.drop(columns=["model_type"]).rename(columns={"padding": "model_name", "data_name": "dataset"})
    df["dataset"] = df["dataset"].str.replace("_", " ")
    df["model_name"] = df["model_name"].str.replace("_", " ").str.capitalize()
    df["model_name"] = df["model_name"].str.replace("Mirror", "Symmetric")
    df["metric"] = df["metric"].str.replace("test/", "")

    return df


models = ["BSARec_Padding", "SASRec", "FMLPRec", "FeaRec"]

# retrieve padding runs
wandb_parser.reset()
df = wandb_parser.register_and_parse(
    project="padding_experiment_final",
    constraints=dict(
        model_type=models,
        attention_probs_dropout_prob=[0.5],
        max_seq_length=[50]
    ),
    cfg=["model_type", "data_name", "seed", "padding", "flip_zero_padding"],
    summary=[f"test/{metric_name}" for metric_name in metrics]
)
df = df[(df["flip_zero_padding"].isnull())]
df = df.drop(columns=["flip_zero_padding"])


for model in models[1:]:
    pseudo_models = ["Zero", "Cyclic", "Reflect", "Symmetric"]

    df_model = df[df["model_type"] == model]
    df_model = standard_post_processing_padding(df_model)
    model_name = model if model != "BSARec_Padding" else "BSARec"

    padding_table_latex, _, _ = create_main_table(
        df_model.copy(deep=True),
        models=pseudo_models,
        metrics=metrics,
        target_model="Zero",
        show_second_best=False,
        table_label=f"tab:padding-{model_name.lower()}",
        table_caption=r"""Performance with different padding methods for """ + model_name + r""". Results are averaged over five runs.
    The best scores are marked bold.
    The column \texttt{Diff.} denotes the relative difference between Zero padding and the best remaining padding method. Statistically significant differences \(p <.05\) are identified using a \(t\)-test and marked with \textsuperscript{*}."""
    )

    print(padding_table_latex)
    print()

# Standard deviations (per dataset)

In [ ]:
df = df_immutable.copy(deep=True).drop(columns=["seed"])

In [ ]:
df.groupby(["model_name", "dataset"]).nunique()

In [ ]:
datasets = df["dataset"].unique()

def format_std_row(row):
    formatted_row = {
        "model_name": row[("model_name", "")],
        "metric": row[("metric", "")],
        "value": f"${row[('value', 'mean')]:.4f} \pm {row[('value', 'std')]:.4f}$"
    }

    return pd.Series(formatted_row)

for dataset in datasets:
    df_subset = df[df["dataset"] == dataset].drop(columns=["dataset"])
    df_subset = (
        df_subset.groupby(["model_name", "metric"])
        .agg(["mean", "std"])
        .reset_index()
        .apply(format_std_row, axis=1)
        .pivot(index="model_name", columns="metric", values="value")
    )

    # sort columns and index
    df_subset = df_subset.reindex(["HR@5", "HR@10", "HR@20", "NDCG@5", "NDCG@10", "NDCG@20"], axis=1)
    df_subset = df_subset.reindex([n.replace("_", " ") for n in models], axis=0)



    # add begin{table} and end{table} tags
    latex = df_subset.to_latex(escape=False, na_rep="", multicolumn=True, multirow=True)
    latex = "\\begin{table*}[h]\n\\centering\n" + latex
    latex += "\\caption{" + f"Standard deviations on the {dataset} dataset. Averaged over 10 runs." + "}\\label{tab:reproduction-std-" + dataset + "}\n"
    latex += "\\end{table*}\n"
    latex = latex.replace("model_name &  &  &  &  &  &  \\\\\n", "")
    print(latex)
    print()


# Wavelet comparison

In [ ]:

wandb_parser = WandbParser(entity="bsarec", api=api, verbose=True)

# models = ["SASRec", "BERT4Rec", "FMLPRec", "duoRec", "FeaRec", "BSARec", "BSARec_Wavelet"]
models = ["BSARec_Wavelet"]
datasets = [
    'LastFM',
    'ML-1M',
]
metrics = ["HR@5", "HR@10", "HR@20", "NDCG@5", "NDCG@10", "NDCG@20"]
seed = None

# add extra runs
wandb_parser.register_project(
    project="Wavelet_Experiment",
    model_type=models[-1:],
    data_name=datasets,
    seed=seed
)

df = wandb_parser.parse(
    cfg=["model_type", "data_name", "seed", "wavelet"],
    summary=[f"test/{metric_name}" for metric_name in metrics],
    post_processing=standard_post_processing
)
df_immutable = df.copy(deep=True)

df.head(5)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# 1) Normalize wavelet names
df_immutable["wavelet"] = df_immutable["wavelet"].str.strip()

# 2) Build a 13-color palette from the “tab20” set
wavelets = df_immutable["wavelet"].unique()
palette = sns.color_palette("tab20", n_colors=len(wavelets))

# 3) Metrics and subplot grid
metrics = ["HR@5","HR@10","HR@20","NDCG@5","NDCG@10","NDCG@20"]
fig, axes = plt.subplots(3, 2, figsize=(14, 18))
axes = axes.flatten()

# placeholders for a single shared legend
legend_handles = legend_labels = None

for ax, metric in zip(axes, metrics):
    dfm = df_immutable[df_immutable["metric"] == metric]
    grouped = (
        dfm
        .groupby(["dataset","model_name","wavelet"])
        .agg(mean_value=("value","mean"),
             std_value=("value","std"))
        .reset_index()
    )

    # 4) Pass the big palette here
    sns.barplot(
        data=grouped,
        x="dataset",
        y="mean_value",
        hue="wavelet",
        palette=palette,
        ci=None,
        ax=ax
    )

    # grab legend info once
    if legend_handles is None:
        legend_handles, legend_labels = ax.get_legend_handles_labels()
    ax.get_legend().remove()

    # 5) Manual error bars
    for i, bar in enumerate(ax.patches):
        h = bar.get_height()
        std = grouped["std_value"].values[i]
        if not np.isnan(std):
            ax.errorbar(
                bar.get_x() + bar.get_width()/2,
                h,
                yerr=std,
                fmt="none",
                c="black",
                capsize=3
            )
        bar.set_edgecolor("black")
        bar.set_linewidth(1)

    ax.set_title(metric)
    ax.set_xlabel("")        # common x-label set below
    ax.set_ylabel(metric)
    ax.tick_params(axis="x", rotation=45)

# 6) Common labels
fig.supxlabel("Dataset", y=0.02)
fig.supylabel("Mean score", x=0.02)

# 7) Single legend at the bottom
fig.legend(
    legend_handles, legend_labels,
    title="Wavelet",
    loc="lower center",
    ncol=7,                # split 13 entries over 2 rows
    frameon=False,
    bbox_to_anchor=(0.5, -0.04),
)

plt.subplots_adjust(hspace=0.4, wspace=0.3, bottom=0.12)
plt.show()
